In [34]:
# Phase 3 — Step 3.3.1
# Candidate monthly observation dates

import pandas as pd

dataset_start = pd.Timestamp("2009-12-01 07:45:00")
dataset_end = pd.Timestamp("2011-12-09 12:50:00")

prediction_horizon = pd.Timedelta(days=60)

# Initial design choice: 90 days of historical coverage
minimum_history = pd.Timedelta(days=90)

earliest_observation = dataset_start + minimum_history
latest_observation = dataset_end - prediction_horizon

observation_dates = pd.date_range(
    start=earliest_observation.normalize(),
    end=latest_observation.normalize(),
    freq="MS"
)

# Exclude any month starts before full history is available
observation_dates = observation_dates[
    observation_dates >= earliest_observation
]

print("BUYERPULSE — OBSERVATION DATE DESIGN")
print("=" * 50)

print("Dataset starts:", dataset_start)
print("Dataset ends:", dataset_end)
print("Minimum historical coverage:", minimum_history.days, "days")
print("Prediction horizon:", prediction_horizon.days, "days")

print("\nEarliest eligible observation:", earliest_observation)
print("Latest eligible observation:", latest_observation)

print("\nCandidate monthly snapshots:", len(observation_dates))
print("First snapshot:", observation_dates.min())
print("Last snapshot:", observation_dates.max())

print("\nCandidate observation dates:")
display(pd.DataFrame({
    "ObservationDate": observation_dates
}))

BUYERPULSE — OBSERVATION DATE DESIGN
Dataset starts: 2009-12-01 07:45:00
Dataset ends: 2011-12-09 12:50:00
Minimum historical coverage: 90 days
Prediction horizon: 60 days

Earliest eligible observation: 2010-03-01 07:45:00
Latest eligible observation: 2011-10-10 12:50:00

Candidate monthly snapshots: 19
First snapshot: 2010-04-01 00:00:00
Last snapshot: 2011-10-01 00:00:00

Candidate observation dates:


,ObservationDate
0,2010-04-01
1,2010-05-01
2,2010-06-01
3,2010-07-01
4,2010-08-01
5,2010-09-01
6,2010-10-01
7,2010-11-01
8,2010-12-01
9,2011-01-01


In [35]:
train_dates = observation_dates[
    observation_dates <= pd.Timestamp("2011-01-01")
]

validation_dates = observation_dates[
    (observation_dates >= pd.Timestamp("2011-04-01")) &
    (observation_dates <= pd.Timestamp("2011-05-01"))
]

test_dates = observation_dates[
    observation_dates >= pd.Timestamp("2011-08-01")
]

horizon = pd.Timedelta(days=60)

assert len(train_dates) == 10
assert len(validation_dates) == 2
assert len(test_dates) == 3

assert train_dates.max() + horizon < validation_dates.min()
assert validation_dates.max() + horizon < test_dates.min()

assert test_dates.max() + horizon <= dataset_end

print("BUYERPULSE — TEMPORAL SPLIT")
print("=" * 42)

for name, dates in [
    ("Training", train_dates),
    ("Validation", validation_dates),
    ("Testing", test_dates),
]:
    print(
        f"{name}: {len(dates)} snapshots | "
        f"{dates.min().date()} → {dates.max().date()}"
    )

print("\nAll temporal boundary checks passed.")

BUYERPULSE — TEMPORAL SPLIT
Training: 10 snapshots | 2010-04-01 → 2011-01-01
Validation: 2 snapshots | 2011-04-01 → 2011-05-01
Testing: 3 snapshots | 2011-08-01 → 2011-10-01

All temporal boundary checks passed.
